In [10]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

In [11]:
PROJECT_ROOT = Path.cwd()

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Either file works. The enriched file is fine even though we will
# discard almost all of its extra fields.
SOURCE_FILE = RAW_DATA_DIR / "games_enriched.json"

CSV_OUTPUT_FILE = (
    PROCESSED_DATA_DIR
    / "steam_games_2022_2025_compact.csv"
)

JSON_OUTPUT_FILE = (
    PROCESSED_DATA_DIR
    / "steam_games_2022_2025_compact.json"
)

print(f"Reading from:\n{SOURCE_FILE}")

Reading from:
d:\Workstation\python\steam-analysis\data\raw\games_enriched.json


In [12]:
with open(
    SOURCE_FILE,
    "r",
    encoding="utf-8"
) as file:
    games = json.load(file)

records = []

for appid, game in games.items():
    records.append({
        "appid": int(appid),
        "name": game.get("name"),
        "release_date": game.get("release_date"),
        "short_description": game.get("short_description", ""),
        "tags": game.get("tags", {}),
        "positive": game.get("positive", 0),
        "negative": game.get("negative", 0),
        "price": game.get("price", 0),
    })



df = pd.DataFrame(records)

print(f"Loaded {len(df):,} Steam entries")
df.head()

Loaded 82,107 Steam entries


,appid,name,release_date,short_description,tags,positive,negative,price
0,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",Yuha! I'll start the broadcast! Hakko's extrem...,[],0,0,8.99
1,3631080,Maze Quest VR,"Apr 24, 2025",Its not just a Maze; its a Quest! Enter the ca...,[],0,0,4.99
2,1654170,Agony VR,"Apr 5, 2023",Agony VR is a first-person survival horror gam...,[],0,0,13.99
3,1934300,Armored Brigade II,"Apr 8, 2025",Armored Brigade II revolutionizes real-time ta...,"{'Simulation': 193, 'Strategy': 186, 'RTS': 16...",117,13,35.99
4,1540330,MUMBA IV: Egypt Jewels,"Dec 13, 2021",Your task in MUMBA IV is to destroy coloured j...,"{'Casual': 64, 'Action': 51, 'Strategy': 42, '...",19,6,0.59


In [13]:
df["release_date"] = pd.to_datetime(
    df["release_date"],
    errors="coerce"
)

df["release_year"] = (
    df["release_date"]
    .dt.year
    .astype("Int64")
)

NUMERIC_COLUMNS = [
    "positive",
    "negative",
    "price"
]

for column in NUMERIC_COLUMNS:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    ).fillna(0)

df["positive"] = df["positive"].clip(lower=0)
df["negative"] = df["negative"].clip(lower=0)
df["price"] = df["price"].clip(lower=0)

df["total_reviews"] = (
    df["positive"]
    + df["negative"]
)

df["review_score"] = np.where(
    df["total_reviews"] > 0,
    100 * df["positive"] / df["total_reviews"],
    np.nan
)

In [14]:
REVIEW_MULTIPLIER = 45

df["estimated_revenue"] = (
    df["total_reviews"]
    * REVIEW_MULTIPLIER
    * df["price"]
)

df[
    [
        "name",
        "release_year",
        "total_reviews",
        "price",
        "estimated_revenue"
    ]
].head()

,name,release_year,total_reviews,price,estimated_revenue
0,버튜버 파라노이아 - Vtuber Paranoia,2024,0,8.99,0.00
1,Maze Quest VR,2025,0,4.99,0.00
2,Agony VR,2023,0,13.99,0.00
3,Armored Brigade II,2025,130,35.99,210541.50
4,MUMBA IV: Egypt Jewels,2021,25,0.59,663.75


In [15]:
def simplify_tags(tags):
    if isinstance(tags, dict):
        return sorted(tags.keys())

    return []


In [16]:
START_YEAR = 2022
END_YEAR = 2025
MIN_REVIEWS = 20

compact_games = df.loc[
    df["release_year"].between(
        START_YEAR,
        END_YEAR,
        inclusive="both"
    )
    & df["total_reviews"].ge(MIN_REVIEWS)
    & df["name"].notna()
    & df["name"].astype(str).str.strip().ne("")
].copy()

compact_games["release_year"] = (
    compact_games["release_year"]
    .astype(int)
)

compact_games["appid"] = (
    compact_games["appid"]
    .astype(int)
)

compact_games["tags"] = (
    compact_games["tags"]
    .apply(simplify_tags)
)

print(
    f"Qualifying games: {len(compact_games):,}"
)

display(
    compact_games
    .groupby("release_year")
    .size()
    .rename("games")
    .to_frame()
)

Qualifying games: 15,521


,games
release_year,
2022,3946
2023,4244
2024,4982
2025,2349


In [17]:
compact_games["revenue_rank_in_year"] = (
    compact_games
    .groupby("release_year")["estimated_revenue"]
    .rank(
        method="first",
        ascending=False
    )
    .astype(int)
)
compact_games["qualifying_games_in_year"] = (
    compact_games
    .groupby("release_year")["appid"]
    .transform("size")
    .astype(int)
)

compact_games["revenue_percentile_in_year"] = (
    100
    * (
        1
        - (
            compact_games["revenue_rank_in_year"] - 1
        )
        / compact_games["qualifying_games_in_year"]
    )
).round(2)

In [18]:
OUTPUT_COLUMNS = [
    "appid",
    "name",
    "short_description",
    "tags",
    "release_year",
    "price",
    "total_reviews",
    "review_score",
    "estimated_revenue",
    "revenue_rank_in_year",
    "qualifying_games_in_year",
    "revenue_percentile_in_year",
]

app_games = (
    compact_games[OUTPUT_COLUMNS]
    .sort_values(
        [
            "release_year",
            "revenue_rank_in_year"
        ]
    )
    .reset_index(drop=True)
)

# Reduce unnecessary decimal noise
app_games["price"] = (
    app_games["price"]
    .round(2)
)

app_games["estimated_revenue"] = (
    app_games["estimated_revenue"]
    .round()
    .astype("int64")
)

app_games["review_score"] = (
    app_games["review_score"]
    .round(2)
)

display(app_games.head(20))

,appid,name,short_description,tags,release_year,price,total_reviews,review_score,estimated_revenue,revenue_rank_in_year,qualifying_games_in_year,revenue_percentile_in_year
0,1245620,ELDEN RING,THE CRITICALLY ACCLAIMED FANTASY ACTION RPG. R...,"[3D, Action, Action RPG, Atmospheric, Characte...",2022,38.99,1056677,92.89,1853992630,1,3946,100.00
1,261550,Mount & Blade II: Bannerlord,"A strategy/action RPG. Create a character, eng...","[Action, Adventure, Character Customization, E...",2022,24.99,264391,87.88,297320899,2,3946,99.97
2,648800,Raft,Raft™ throws you and your friends into an epic...,"[Action, Adventure, Base-Building, Building, C...",2022,13.39,346359,93.31,208698615,3,3946,99.95
3,534380,Dying Light 2 Stay Human: Reloaded Edition,Humanity is fighting a losing battle against t...,"[Action, Action RPG, Action-Adventure, Adventu...",2022,17.99,194830,79.11,157724626,4,3946,99.92
4,1593500,God of War,His vengeance against the Gods of Olympus year...,"[3D, Action, Action RPG, Adventure, Atmospheri...",2022,19.99,158017,96.01,142144192,5,3946,99.90
5,1332010,Stray,"Lost, alone and separated from family, a stray...","[Action, Adventure, Atmospheric, Beautiful, Ca...",2022,17.99,156607,97.26,126781197,6,3946,99.87
6,1817070,Marvel’s Spider-Man Remastered,"In Marvel’s Spider-Man Remastered, the worlds ...","[Action, Action-Adventure, Adventure, Atmosphe...",2022,23.99,112392,95.90,121332784,7,3946,99.85
7,1544020,The Callisto Protocol™,Survive to escape the horrors of Callisto and ...,"[3D, Action, Action-Adventure, Adventure, Cine...",2022,59.99,39430,64.93,106443256,8,3946,99.82
8,1687950,Persona 5 Royal,Don the mask and join the Phantom Thieves of H...,"[Adventure, Anime, Colorful, Dating Sim, Detec...",2022,17.99,103144,96.62,83500225,9,3946,99.80
9,1142710,Total War: WARHAMMER III,The cataclysmic conclusion to the Total War: W...,"[Action, Atmospheric, Co-op, Colorful, Dark Fa...",2022,14.99,119858,72.70,80850214,10,3946,99.77


In [19]:
assert app_games["appid"].notna().all()
assert app_games["appid"].is_unique

assert app_games["release_year"].between(
    START_YEAR,
    END_YEAR
).all()

assert app_games["total_reviews"].ge(
    MIN_REVIEWS
).all()

assert app_games["estimated_revenue"].ge(0).all()

assert app_games["revenue_rank_in_year"].ge(1).all()

print("All validation checks passed.")

All validation checks passed.


In [20]:
rank_validation = (
    app_games
    .groupby("release_year")
    .agg(
        games=("appid", "size"),
        lowest_rank=("revenue_rank_in_year", "min"),
        highest_rank=("revenue_rank_in_year", "max"),
        unique_ranks=("revenue_rank_in_year", "nunique"),
        minimum_revenue=("estimated_revenue", "min"),
        maximum_revenue=("estimated_revenue", "max"),
    )
)

display(rank_validation)

,games,lowest_rank,highest_rank,unique_ranks,minimum_revenue,maximum_revenue
release_year,,,,,,
2022,3946,1,3946,3946,0,1853992630
2023,4244,1,4244,4244,0,1539998252
2024,4982,1,4982,4982,0,3104747056
2025,2349,1,2349,2349,0,378305701


In [21]:
app_games.to_csv(
    CSV_OUTPUT_FILE,
    index=False,
    encoding="utf-8"
)

print(f"CSV written to:\n{CSV_OUTPUT_FILE}")

CSV written to:
d:\Workstation\python\steam-analysis\data\processed\steam_games_2022_2025_compact.csv


In [22]:
json_records = app_games.to_dict(
    orient="records"
)

with open(
    JSON_OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        json_records,
        file,
        ensure_ascii=False,
        separators=(",", ":")
    )

print(f"JSON written to:\n{JSON_OUTPUT_FILE}")

JSON written to:
d:\Workstation\python\steam-analysis\data\processed\steam_games_2022_2025_compact.json


In [23]:
def format_file_size(size_bytes):
    if size_bytes >= 1_000_000:
        return f"{size_bytes / 1_000_000:.2f} MB"

    if size_bytes >= 1_000:
        return f"{size_bytes / 1_000:.1f} KB"

    return f"{size_bytes} bytes"


csv_size = CSV_OUTPUT_FILE.stat().st_size
json_size = JSON_OUTPUT_FILE.stat().st_size

print(
    f"CSV:  {format_file_size(csv_size)}"
)

print(
    f"JSON: {format_file_size(json_size)}"
)

CSV:  8.27 MB
JSON: 11.06 MB


In [24]:
csv_check = pd.read_csv(
    CSV_OUTPUT_FILE
)

with open(
    JSON_OUTPUT_FILE,
    "r",
    encoding="utf-8"
) as file:
    json_check = json.load(file)

assert len(csv_check) == len(app_games)
assert len(json_check) == len(app_games)

assert set(csv_check["appid"]) == set(
    app_games["appid"]
)

print(
    f"Verified {len(app_games):,} records "
    "in both output files."
)

Verified 15,521 records in both output files.
